# ColumnTransformer: End-to-End Feature Preprocessing
This notebook demonstrates how to preprocess mixed data types (numerical + categorical) for machine learning.

## Learning Goals
1. Understand **why preprocessing is required** before model training.
2. Compare a **manual preprocessing pipeline** (without `ColumnTransformer`) with an integrated pipeline.
3. See how `ColumnTransformer` makes preprocessing **cleaner, safer, and easier to maintain**.

## Dataset Context
We use a toy COVID dataset with mixed columns:
- Numerical: `age`, `fever` (contains missing values)
- Categorical: `cough` (ordinal), `gender`, `city` (nominal)
- Target: `has_covid`

## Why this notebook has two parts
- **Part A (Without ColumnTransformer):** helps you understand every transformation step manually.
- **Part B (With ColumnTransformer):** shows the production-friendly way to apply the same logic in one reusable pipeline.

<!-- auto-doc -->
### Step 1: Import Required Libraries

**Concept:** Setting up essential libraries for data manipulation and preprocessing.

**What this cell does:** Imports NumPy for numerical arrays and Pandas for data frame operations.

**Code focus:** `import numpy as np` and `import pandas as pd`

**Expected outcome:** Successfully import libraries without errors. These are foundational tools for data preprocessing.

In [75]:
import numpy as np
import pandas as pd

<!-- auto-doc -->
### Step 2: Import Preprocessing Transformers

**Concept:** Importing sklearn transformers for imputation, encoding, and pipeline composition.

**What this cell does:** Imports SimpleImputer for handling missing values, OneHotEncoder and OrdinalEncoder for categorical encoding.

**Code focus:** `from sklearn.impute import SimpleImputer` and encoding classes.

**Expected outcome:** All preprocessing modules are available for use in the pipeline.

In [76]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import OrdinalEncoder

<!-- auto-doc -->
### Step 3: Load the COVID Dataset

**Concept:** Loading a CSV file containing mixed data types (numerical and categorical columns).

**What this cell does:** Reads the 'covid_toy.csv' file into a Pandas DataFrame for exploration and preprocessing.

**Code focus:** `pd.read_csv('covid_toy.csv')`

**Expected outcome:** DataFrame loaded successfully with all rows and columns from the CSV file.

In [77]:
df = pd.read_csv('covid_toy.csv')

<!-- auto-doc -->
### Step 4: Explore Dataset Structure

**Concept:** Inspecting the first few rows to understand data types and column characteristics.

**What this cell does:** Displays the first few rows of the DataFrame to visualize the structure of features and identify mixed data types.

**Code focus:** `df.head()`

**Expected outcome:** View columns (age, fever, cough, gender, city, has_covid) with sample data to understand the dataset.

In [27]:
df.head()

,age,gender,fever,cough,city,has_covid
0,60,Male,103.0,Mild,Kolkata,No
1,27,Male,100.0,Mild,Delhi,Yes
2,42,Male,101.0,Mild,Delhi,No
3,31,Female,98.0,Mild,Kolkata,No
4,65,Female,101.0,Mild,Mumbai,No


<!-- auto-doc -->
### Step 5: Check for Missing Values

**Concept:** Identifying columns with null/missing values to understand data quality.

**What this cell does:** Counts the number of missing values in each column to determine which columns require imputation.

**Code focus:** `df.isnull().sum()`

**Expected outcome:** Shows that 'fever' column has missing values (NaN) that need to be handled during preprocessing.

In [80]:
df.isnull().sum()

age           0
gender        0
fever        10
cough         0
city          0
has_covid     0
dtype: int64

<!-- auto-doc -->
### Step 6: Split Data into Train and Test Sets

**Concept:** Dividing data into training and test subsets to avoid data leakage during preprocessing.

**What this cell does:** Performs an 80-20 train-test split on features (X) and target (y) to establish separate datasets.

**Code focus:** `train_test_split()` with test_size=0.2

**Expected outcome:** X_train, X_test, y_train, y_test sets created. Training set is larger for model fitting, test set for evaluation.

In [29]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test = train_test_split(df.drop(columns=['has_covid']),df['has_covid'],
                                                test_size=0.2)

<!-- auto-doc -->
### Step 7: Inspect Training Data Structure

**Concept:** Verifying the shape and content of the training feature set.

**What this cell does:** Displays the X_train DataFrame to confirm it has all features (excluding target) and correct dimensions.

**Code focus:** `X_train`

**Expected outcome:** View training set with features: age, fever, cough, gender, city. Ready for manual preprocessing in Part A.

In [81]:
X_train

,age,gender,fever,cough,city
55,81,Female,101.0,Mild,Mumbai
76,80,Male,100.0,Mild,Bangalore
22,71,Female,98.0,Strong,Kolkata
93,27,Male,100.0,Mild,Kolkata
33,26,Female,98.0,Mild,Kolkata
...,...,...,...,...,...
2,42,Male,101.0,Mild,Delhi
51,11,Female,100.0,Strong,Kolkata
5,84,Female,NaN,Mild,Bangalore
40,49,Female,102.0,Mild,Delhi


<!-- auto-doc -->
### Step 8: View Training Feature Set

**Concept:** Displaying the complete training dataset to understand column structure before preprocessing.

**What this cell does:** Shows the X_train DataFrame with all features for visual inspection.

**Code focus:** `X_train`

**Expected outcome:** DataFrame displayed with all rows and feature columns ready for preprocessing.

## Part A: Without ColumnTransformer (Manual Pipeline)
In this section, we preprocess each column group separately and then combine all transformed arrays.

### Why this is done
- To understand exactly what each transformer does.
- To see the bookkeeping effort required when transformations are applied manually.
- To appreciate where mistakes can happen (column order, repeated code, train/test handling).

### Steps covered
1. Impute missing values in `fever`.
2. Apply ordinal encoding on `cough` (`Mild < Strong`).
3. Apply one-hot encoding on `gender` and `city`.
4. Keep `age` as a passthrough numerical feature.
5. Concatenate all transformed outputs into a final feature matrix.

<!-- auto-doc -->
### Step 9: Handle Missing Values with SimpleImputer

**Concept:** Filling missing values in numerical columns using imputation strategies.

**What this cell does:** Applies SimpleImputer to the 'fever' column with default strategy (mean imputation) on both train and test sets.

**Code focus:** `SimpleImputer()` with `fit_transform()` and `transform()`

**Expected outcome:** X_train_fever and X_test_fever arrays with no missing values. Shape shows (n_samples, 1).

In [83]:
# adding simple imputer to fever col
si = SimpleImputer()
X_train_fever = si.fit_transform(X_train[['fever']])

# also the test data
X_test_fever = si.fit_transform(X_test[['fever']])
                                 
X_train_fever.shape

(80, 1)

<!-- auto-doc -->
### Step 10: Encode Ordinal Categorical Feature

**Concept:** Converting ordinal categorical features (with meaningful order) into numerical values.

**What this cell does:** Applies OrdinalEncoder to 'cough' column with defined order (Mild < Strong), transforming both train and test data.

**Code focus:** `OrdinalEncoder(categories=[['Mild','Strong']])`

**Expected outcome:** X_train_cough and X_test_cough arrays with numerical values [0, 1] representing the ordinal relationship.

In [85]:
# Ordinalencoding -> cough
oe = OrdinalEncoder(categories=[['Mild','Strong']])
X_train_cough = oe.fit_transform(X_train[['cough']])

# also the test data
X_test_cough = oe.fit_transform(X_test[['cough']])

X_train_cough.shape

(80, 1)

<!-- auto-doc -->
### Step 11: Encode Nominal Categorical Features

**Concept:** Converting nominal categorical features (without inherent order) using one-hot encoding.

**What this cell does:** Applies OneHotEncoder to 'gender' and 'city' columns, dropping the first category to avoid multicollinearity.

**Code focus:** `OneHotEncoder(drop='first', sparse=False)`

**Expected outcome:** X_train_gender_city and X_test_gender_city arrays with binary columns for each category, sparse=False for dense array format.

In [87]:
# OneHotEncoding -> gender,city
ohe = OneHotEncoder(drop='first',sparse=False)
X_train_gender_city = ohe.fit_transform(X_train[['gender','city']])

# also the test data
X_test_gender_city = ohe.fit_transform(X_test[['gender','city']])

X_train_gender_city.shape

(80, 4)

<!-- auto-doc -->
### Step 12: Extract Numerical Features

**Concept:** Isolating numerical features that do not require transformation (passthrough).

**What this cell does:** Extracts the 'age' column by dropping all categorical and imputed columns from both train and test sets.

**Code focus:** `drop()` with list of non-numerical columns

**Expected outcome:** X_train_age and X_test_age arrays containing only the age feature, ready for concatenation.

In [89]:
# Extracting Age
X_train_age = X_train.drop(columns=['gender','fever','cough','city']).values

# also the test data
X_test_age = X_test.drop(columns=['gender','fever','cough','city']).values

X_train_age.shape

(80, 1)

<!-- auto-doc -->
### Step 13: Concatenate All Transformed Features

**Concept:** Combining all individually transformed feature arrays into a single feature matrix.

**What this cell does:** Uses NumPy concatenate to merge age, fever, gender_city, and cough features along axis 1 (columns).

**Code focus:** `np.concatenate()` with axis=1

**Expected outcome:** X_train_transformed and X_test_transformed with shape (n_samples, n_total_features) - all features combined into one array.

In [92]:
X_train_transformed = np.concatenate((X_train_age,X_train_fever,X_train_gender_city,X_train_cough),axis=1)
# also the test data
X_test_transformed = np.concatenate((X_test_age,X_test_fever,X_test_gender_city,X_test_cough),axis=1)

X_train_transformed.shape

(80, 7)

<!-- auto-doc -->
### Step 14: Import ColumnTransformer

**Concept:** Using sklearn's ColumnTransformer for cleaner, automated preprocessing pipeline.

**What this cell does:** Imports the ColumnTransformer class that will consolidate all transformations into a single reusable object.

**Code focus:** `from sklearn.compose import ColumnTransformer`

**Expected outcome:** ColumnTransformer class is now available for building the integrated pipeline in Part B.

## Part B: With ColumnTransformer (Recommended Pipeline)
Now we apply the same preprocessing logic using a single `ColumnTransformer` object.

### Why this is done
- Centralizes all preprocessing in one place.
- Preserves a consistent transformation order across training and test data.
- Reduces repetitive code and improves readability.
- Makes deployment and model pipelines easier to build and maintain.

### Expected Outcome
The transformed output should contain the same feature information as Part A, but created through a cleaner, reusable pipeline.

<!-- auto-doc -->
### Step 15: Define ColumnTransformer Pipeline

**Concept:** Creating a unified preprocessing pipeline with multiple transformers applied to specific column groups.

**What this cell does:** Defines a ColumnTransformer with three transformers: imputation for 'fever', ordinal encoding for 'cough', and one-hot encoding for 'gender' and 'city'. remainder='passthrough' keeps other columns unchanged.

**Code focus:** `ColumnTransformer(transformers=[...], remainder='passthrough')`

**Expected outcome:** 'transformer' object created, ready to fit and transform data in a single, consistent operation.

In [32]:
from sklearn.compose import ColumnTransformer

<!-- auto-doc -->
### Step 16: Fit and Transform Training Data

**Concept:** Learning transformation parameters from training data and applying transformations simultaneously.

**What this cell does:** Uses fit_transform() on X_train to learn imputation statistics, encoding mappings, and apply transformations in one step.

**Code focus:** `transformer.fit_transform(X_train).shape`

**Expected outcome:** Shape output shows (n_samples, n_features) confirming all transformations applied successfully on training set.

In [95]:
transformer = ColumnTransformer(transformers=[
    ('tnf1',SimpleImputer(),['fever']),
    ('tnf2',OrdinalEncoder(categories=[['Mild','Strong']]),['cough']),
    ('tnf3',OneHotEncoder(sparse=False,drop='first'),['gender','city'])
],remainder='passthrough')

<!-- auto-doc -->
### Step 17: Transform Test Data Using Learned Parameters

**Concept:** Applying the same learned transformations to test data without refitting (preventing data leakage).

**What this cell does:** Calls transform() on X_test to apply previously learned imputation, encoding, and other transformations.

**Code focus:** `transformer.transform(X_test).shape`

**Expected outcome:** Shape output shows (n_test_samples, n_features) confirming test set transformed consistently with training set using same parameters.

In [97]:
transformer.fit_transform(X_train).shape

(80, 7)

<!-- auto-doc -->
### Step 18: Summary and Comparison

**Concept:** Understanding the trade-offs between manual and automated preprocessing approaches.

**What this cell does:** Displays reference information comparing the manual pipeline (Part A) with the ColumnTransformer approach (Part B).

**Code focus:** Text and conceptual comparison

**Expected outcome:** Clear understanding that both approaches produce the same transformed features, but ColumnTransformer is more scalable and maintainable for production workflows.

In [99]:
transformer.transform(X_test).shape

(20, 7)

## Final Comparison: Manual vs ColumnTransformer
### Without ColumnTransformer
- More explicit, useful for learning internals.
- More code, more chances of inconsistency.

### With ColumnTransformer
- Concise and structured.
- Better for real ML workflows and reproducibility.

### Key Takeaway
Use the manual approach to learn concepts, then switch to `ColumnTransformer` for robust and scalable preprocessing.